# Tag 3 - Data Preparation

## Ladezelle

In [40]:
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split

SQL_PATH = "../data/interim/projekt.sqlite"

con = sqlite3.connect(SQL_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)
blockgroups = pd.read_sql("SELECT * FROM blockgruppen", con)
X = blockgroups.drop(columns=["MedHouseVal"])
y = blockgroups["MedHouseVal"]

print(blockgroups.shape)
# -> (20640, 9)

(20640, 9)


## Train-Test-Split


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Trainingszeilen:", len(X_train))
print("Testzeilen:", len(X_test))

Trainingszeilen: 16512
Testzeilen: 4128


## Gedeckelte Werte

In [42]:
grenze_medhouseval = 5.00001
grenze_houseage = 52.0
anzahl_medhouseval = (y_train >= grenze_medhouseval).sum()
anzahl_houseage = (X_train["HouseAge"] >= grenze_houseage).sum()

print("Gedeckelte Werte aus MedHouseVal:", anzahl_medhouseval)
print("Gedeckelte Werte aus HouseAge:", anzahl_houseage)

Gedeckelte Werte aus MedHouseVal: 786
Gedeckelte Werte aus HouseAge: 1023


## Ausreißer

In [43]:
train = pd.concat([X_train, y_train], axis=1)
columns = train.select_dtypes(include="number")
q1 = 0
q3 = 0
iqr = 0

# display(columns)
# print(columns.shape)
for column in columns:
    print(f"---{column}---")
    q1 = columns[column].quantile(0.25)
    q3 = columns[column].quantile(0.75)
    iqr = q3 - q1
    untergrenze = q1 - 1.5 * iqr
    obergrenze = q3 + 1.5 * iqr
    ausreisser = columns[(columns[column] < untergrenze) |
                          (columns[column] > obergrenze)]

    print(f"Q1: {q1:.2f}")
    print(f"Q3: {q3:.2f}")
    print(f"Anzahl Ausreißer: {len(ausreisser)}")
    print(f"Anteil: {len(ausreisser) / len(columns) * 100:.2f} %\n")

---MedInc---
Q1: 2.57
Q3: 4.77
Anzahl Ausreißer: 529
Anteil: 3.20 %

---HouseAge---
Q1: 18.00
Q3: 37.00
Anzahl Ausreißer: 0
Anteil: 0.00 %

---AveRooms---
Q1: 4.45
Q3: 6.06
Anzahl Ausreißer: 410
Anteil: 2.48 %

---AveBedrms---
Q1: 1.01
Q3: 1.10
Anzahl Ausreißer: 1153
Anteil: 6.98 %

---Population---
Q1: 789.00
Q3: 1726.00
Anzahl Ausreißer: 955
Anteil: 5.78 %

---AveOccup---
Q1: 2.43
Q3: 3.28
Anzahl Ausreißer: 582
Anteil: 3.52 %

---Latitude---
Q1: 33.93
Q3: 37.72
Anzahl Ausreißer: 0
Anteil: 0.00 %

---Longitude---
Q1: -121.81
Q3: -118.01
Anzahl Ausreißer: 0
Anteil: 0.00 %

---MedHouseVal---
Q1: 1.20
Q3: 2.65
Anzahl Ausreißer: 870
Anteil: 5.27 %



# Tag 4: Baseline und erstes Modell

## Erstes Modell

In [44]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

dummy = DummyRegressor(strategy="mean")
linear = LinearRegression()
dummy.fit(X_train, y_train)
linear.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_pred_linear = linear.predict(X_test)
 
print("Erste 5 Vorhersagen Dummy:")
print(y_pred_dummy[:5])
 
print("Erste 5 Vorhersagen Linear Regression:")
print(y_pred_linear[:5])

Erste 5 Vorhersagen Dummy:
[2.07194694 2.07194694 2.07194694 2.07194694 2.07194694]
Erste 5 Vorhersagen Linear Regression:
[0.71912284 1.76401657 2.70965883 2.83892593 2.60465725]


## Metriken berechnen

In [45]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

modelle = {
    "DummyRegressor": dummy,
    "LinearRegression": linear
}
ergebnisse = []

for name, modell in modelle.items():
    y_pred = modell.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    ergebnisse.append({
        "Modell": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })
# -----------------------------------------------------------------------
# | Metrik | Was misst sie?	                                  | Optimal |
# | MAE	   | Durchschnittliche absolute Abweichung	          | 0       |
# | RMSE   | Fehlermaß mit stärkerer Gewichtung großer Fehler | 0       |
# | R²	   | Modellgüte relativ zur Mittelwertvorhersage	  | 1       |
# -----------------------------------------------------------------------

vergleich = pd.DataFrame(ergebnisse)
display(vergleich.round(4))

,Modell,MAE,RMSE,R2
0,DummyRegressor,0.9061,1.1449,-0.0002
1,LinearRegression,0.5332,0.7456,0.5758


## Prozentuale Verbesserung

In [46]:
mae_dummy = vergleich.loc[vergleich["Modell"] == "DummyRegressor", "MAE"].iloc[0]
mae_linear = vergleich.loc[vergleich["Modell"] == "LinearRegression", "MAE"].iloc[0]

verbesserung = (mae_dummy - mae_linear) / mae_dummy * 100
mae_usd = mae_linear * 100_000

print(f"Verbesserung gegenüber der Baseline: {verbesserung:.2f}%")
print(f"MAE der LinearRegression: {mae_usd:,.2f} USD")

Verbesserung gegenüber der Baseline: 41.15%
MAE der LinearRegression: 53,320.01 USD


# Tag 5 - Zweites Modell & Feature Engineering

## Zweites Modell

In [47]:
from sklearn.tree import DecisionTreeRegressor

baum = DecisionTreeRegressor(max_depth=6, random_state=42)

baum.fit(X_train, y_train)
vorhersage_baum = baum.predict(X_test)
print(vorhersage_baum[:5])

[1.11480881 1.14983053 3.78527446 2.31297199 1.51260182]


## Vergleich mit BaseLine und LinearRegression

In [48]:
from sklearn.model_selection import cross_val_score

modelle["DecisionTreeRegressor"] = baum
ergebnisse_cv = []

for name, modell in modelle.items():
    # 5-fache Kreuzvalidierung auf den Trainingsdaten mit negativem MAE
    scores = cross_val_score(modell, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    mae_werte = -scores

    ergebnisse_cv.append({
        "Modell": name,
        "MAE Mittelwert": mae_werte.mean(),
        "MAE Standardabweichung": mae_werte.std()
    })

vergleich_cv = pd.DataFrame(ergebnisse_cv)
display(vergleich_cv.round(4))

,Modell,MAE Mittelwert,MAE Standardabweichung
0,DummyRegressor,0.9139,0.0085
1,LinearRegression,0.5291,0.0053
2,DecisionTreeRegressor,0.4864,0.0126


### Modellauswahl

Der `DecisionTreeRegressor` erreicht mit einem MAE-Mittelwert von 0,4864 den niedrigsten Fehler. Die `LinearRegression` erreicht 0,5291 und der DummyRegressor 0,9139.

Der Entscheidungsbaum verbessert den MAE gegenüber der Baseline um etwa 46,8 %.
Allerdings weist er eine etwas höhere Standardabweichung auf als die `LinearRegression`.
Daher wird der `DecisionTreeRegressor` zunächst als bevorzugtes Modell ausgewählt.
 

## Feature Engineering (light)

### Schiefe und Wertebereiche

In [ ]:
for spalte in X_train.select_dtypes(include="number"):
    print(f"--- {spalte} ---")
    print(f"Schiefe: {X_train[spalte].skew():.2f}")
    print(f"Minimum: {X_train[spalte].min():.2f}")
    print(f"Maximum: {X_train[spalte].max():.2f}\n")
    
    # --------------------------
    # | Schiefe | Bedeutung    |
    # |   ~ 0   | symmetrisch  |
    # |   > 0   | rechtsschief |
    # |   < 0   | linksschief  |
    # --------------------------

--- MedInc ---
Schiefe: 1.63
Minimum: 0.50
Maximum: 15.00

--- HouseAge ---
Schiefe: 0.06
Minimum: 1.00
Maximum: 52.00

--- AveRooms ---
Schiefe: 18.61
Minimum: 0.89
Maximum: 141.91

--- AveBedrms ---
Schiefe: 23.17
Minimum: 0.33
Maximum: 25.64

--- Population ---
Schiefe: 5.28
Minimum: 3.00
Maximum: 35682.00

--- AveOccup ---
Schiefe: 88.05
Minimum: 0.69
Maximum: 1243.33

--- Latitude ---
Schiefe: 0.46
Minimum: 32.55
Maximum: 41.95

--- Longitude ---
Schiefe: -0.29
Minimum: -124.35
Maximum: -114.31



#### Auswertung

| Merkmal | Schiefe | Minimum | Maximum | Einschätzung |
|---|---:|---:|---:|---|
| MedInc | 1,63 | 0,50 | 15,00 | Deutlich rechtsschief |
| HouseAge | 0,06 | 1,00 | 52,00 | Nahezu symmetrisch |
| AveRooms | 18,61 | 0,89 | 141,91 | Extrem rechtsschief |
| AveBedrms | 23,17 | 0,33 | 25,64 | Extrem rechtsschief |
| Population | 5,28 | 3,00 | 35.682,00 | Stark rechtsschief |
| AveOccup | 88,05 | 0,69 | 1.243,33 | Extrem rechtsschief |
| Latitude | 0,46 | 32,55 | 41,95 | Leicht rechtsschief |
| Longitude | -0,29 | -124,35 | -114,31 | Leicht linksschief |

**Fazit:** Besonders `AveRooms`, `AveBedrms`, `Population` und `AveOccup` weisen eine starke Rechtsschiefe und teilweise extreme Maximalwerte auf. Da alle Werte dieser Merkmale positiv sind, eignen sie sich für einen Versuch mit einer Log-Transformation. Anschließend wird mithilfe der Kreuzvalidierung geprüft, ob sich dadurch die Vorhersagequalität verbessert.

### Log-Transformation

In [50]:
X_train_log = X_train.copy()

log_spalten = ["AveRooms", "AveBedrms", "Population", "AveOccup"]

for spalte in log_spalten:
    X_train_log[spalte] = np.log(X_train_log[spalte])
    print(f"--- {spalte} ---")
    print(f"Schiefe: {X_train_log[spalte].skew():.2f}")
    print(f"Minimum: {X_train_log[spalte].min():.2f}")
    print(f"Maximum: {X_train_log[spalte].max():.2f}\n")

ergebnisse_log = []

for name, modell in modelle.items():
    scores = cross_val_score(
        modell,
        X_train_log,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error"
    )

    mae_werte = -scores

    ergebnisse_log.append({
        "Modell": name,
        "MAE Mittelwert": mae_werte.mean(),
        "MAE Standardabweichung": mae_werte.std()
    })

vergleich_log = pd.DataFrame(ergebnisse_log)
display(vergleich_log.round(4))

# ------------------------------------------------------
# Log-Transformation:
# - Komprimiert große Werte stärker als kleine Werte
# - Kann die Rechtsschiefe einer Verteilung reduzieren
# - np.log() ist nur für positive Werte definiert
# - Kann besonders linearen Modellen helfen
# - Ob das Modell besser wird, zeigt die Cross-Validation
# ------------------------------------------------------

--- AveRooms ---
Schiefe: 0.83
Minimum: -0.12
Maximum: 4.96

--- AveBedrms ---
Schiefe: 6.23
Minimum: -1.10
Maximum: 3.24

--- Population ---
Schiefe: -1.06
Minimum: 1.10
Maximum: 10.48

--- AveOccup ---
Schiefe: 2.35
Minimum: -0.37
Maximum: 7.13



,Modell,MAE Mittelwert,MAE Standardabweichung
0,DummyRegressor,0.9139,0.0085
1,LinearRegression,0.4836,0.0077
2,DecisionTreeRegressor,0.4865,0.0127


### Gegenüberstellung

In [51]:
vergleich_gesamt = pd.concat([
    vergleich_cv.assign(Merkmalsvariante="Rohmerkmale"),
    vergleich_log.assign(Merkmalsvariante="Log-Merkmale")
], ignore_index=True)

vergleich_gesamt = vergleich_gesamt[
    ["Modell", "Merkmalsvariante", "MAE Mittelwert", "MAE Standardabweichung"]
]

display(vergleich_gesamt.round(4))

,Modell,Merkmalsvariante,MAE Mittelwert,MAE Standardabweichung
0,DummyRegressor,Rohmerkmale,0.9139,0.0085
1,LinearRegression,Rohmerkmale,0.5291,0.0053
2,DecisionTreeRegressor,Rohmerkmale,0.4864,0.0126
3,DummyRegressor,Log-Merkmale,0.9139,0.0085
4,LinearRegression,Log-Merkmale,0.4836,0.0077
5,DecisionTreeRegressor,Log-Merkmale,0.4865,0.0127


#### Ergebnis des Feature Engineerings

Die Log-Transformation der vier stark rechtsschiefen Merkmale `AveRooms`, `AveBedrms`, `Population` und `AveOccup` verbessert insbesondere die `LinearRegression`. Ihr durchschnittlicher MAE sinkt von 0,5291 auf 0,4836 (ca. 8,6 %).

Beim `DecisionTreeRegressor` bleibt der MAE nahezu unverändert (0,4864 gegenüber 0,4865).

Die `LinearRegression` mit Log-Merkmalen erreicht damit den niedrigsten durchschnittlichen MAE. Der Unterschied zum `DecisionTreeRegressor` ist allerdings gering und liegt unter der Streuung der Cross-Validation. Daher wird die `LinearRegression` mit Log-Merkmalen für die abschließende Bewertung bevorzugt.